In [2]:
import os
from dotenv import load_dotenv
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS


In [3]:
# Ładowanie zmiennych środowiskowych z pliku .env
load_dotenv()

# Pobieranie klucza API z pliku .env
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL= os.getenv("MODEL")
YOUTUBE_VIDEO = "https://www.youtube.com/watch?v=0mlEHuyjWes"
ONEDRIVE_PATH_PDF = "C:\\Users\\Agnieszka\\OneDrive\\faiss_index"
ONEDRIVE_PATH_YOUTUBE = "C:\\Users\\Agnieszka\\OneDrive\\youtube_faiss_index"

# Inicjalizacja modelu GPT i osadzania (embeddings) od OpenAI
model = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model=MODEL)
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

In [ ]:
def load_or_create_embeddings(source_text_chunks, onedrive_path):
    if os.path.exists(onedrive_path):
        print(f"Ładowanie istniejących embeddingów z: {onedrive_path}")
        # Ładowanie zapisanych embeddingów FAISS
        vectorstore = FAISS.load_local(onedrive_path, embeddings)
    else:
        print(f"Embeddingi nie istnieją. Generowanie nowych embeddingów i zapis do: {onedrive_path}")
        # Generowanie nowych embeddingów
        vectorstore = FAISS.from_texts(source_text_chunks, embeddings)
        # Zapisanie embeddingów FAISS na dysku
        vectorstore.save_local(onedrive_path)
    return vectorstore


In [ ]:
import fitz  # PyMuPDF
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Ścieżka do folderu z plikami PDF
PDF_FOLDER = "D:\\data science w medycynie\\projekt grupowy na danych tekstowych\\ChPL_1000"

# Zmienna do przechowywania tekstu ze wszystkich plików PDF
all_text_pdf = ""

# Iterowanie po wszystkich plikach PDF w folderze
for filename in os.listdir(PDF_FOLDER):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(PDF_FOLDER, filename)
        
        try:
            # Otwieranie PDF za pomocą PyMuPDF (fitz)
            doc = fitz.open(pdf_path)
            
            # Iterowanie po stronach i zbieranie tekstu
            for page_num in range(doc.page_count):
                page = doc.load_page(page_num)
                all_text_pdf += page.get_text()  # Pobieranie tekstu z każdej strony
                
        except Exception as e:
            print(f"Błąd podczas przetwarzania pliku {filename}: {e}")

# Dzielimy tekst z PDF na chunki
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
chunks_pdf = splitter.split_text(all_text_pdf)


print(f"Liczba chunków w pdf: {len(chunks_pdf)}")

# Ładowanie lub generowanie embeddingów dla PDF
pdf_vectorstore = load_or_create_embeddings(chunks_pdf, ONEDRIVE_PATH_PDF)


In [ ]:
import os
import tempfile
import whisper
from pytube import YouTube

# URL do wideo YouTube
YOUTUBE_VIDEO = "https://www.youtube.com/watch?v=0mlEHuyjWes"

# Ścieżka do OneDrive, gdzie zapiszesz transkrypcję
TRANSCRIPTION_PATH = "C:\\Users\\Agnieszka\\OneDrive\\transcription.txt"

# Inicjalizacja modelu Whisper
whisper_model = whisper.load_model("base")

# Sprawdzenie, czy już istnieje plik z transkrypcją na OneDrive
if not os.path.exists(TRANSCRIPTION_PATH):
    youtube = YouTube(YOUTUBE_VIDEO)
    audio = youtube.streams.filter(only_audio=True).first()

    with tempfile.TemporaryDirectory() as tmpdir:
        audio_file = audio.download(output_path=tmpdir)
        transcription = whisper_model.transcribe(audio_file, fp16=False)["text"].strip()

        # Zapisanie transkrypcji do pliku na OneDrive
        with open(TRANSCRIPTION_PATH, "w") as file:
            file.write(transcription)
        print(f"Transkrypcja zapisana do: {TRANSCRIPTION_PATH}")

else:
    print(f"Transkrypcja już istnieje w: {TRANSCRIPTION_PATH}")

# Odczytanie transkrypcji z pliku na OneDrive
with open(TRANSCRIPTION_PATH, "r") as file:
    youtube_text = file.read()

# Wyświetlenie treści transkrypcji
print(f"Treść transkrypcji:\n{youtube_text}")



In [ ]:

# Dzielimy tekst z transkrypcji YouTube na chunki
chunks_youtube = splitter.split_text(youtube_text)


print(f"Liczba chunków z youtube: {len(chunks_youtube)}")

# Ładowanie lub generowanie embeddingów dla YouTube
youtube_vectorstore = load_or_create_embeddings(chunks_youtube, ONEDRIVE_PATH_YOUTUBE)

In [ ]:
from langchain.prompts import PromptTemplate
from operator import itemgetter

# Nowy szablon, który uwzględnia źródło danych (PDF, YouTube, oba)
template = """
You are an assistant that provides answers to questions based on
a given context. You can retrieve information from different sources like PDF documents or YouTube transcriptions.

Answer the question based on the given context. If you can't answer the
question, reply "I don't know".

Be as concise as possible and go straight to the point.

Context: {context}

Question: {question}
Source: {source}
"""

prompt = PromptTemplate.from_template(template)


In [ ]:
def retrieve_context(question, source, pdf_vectorstore, youtube_vectorstore):
    if source == "pdf":
        # Przeszukiwanie tylko w bazie PDF
        context = pdf_vectorstore.similarity_search(question)
    elif source == "youtube":
        # Przeszukiwanie tylko w bazie YouTube
        context = youtube_vectorstore.similarity_search(question)
    else:
        # Przeszukiwanie obu baz
        pdf_context = pdf_vectorstore.similarity_search(question)
        youtube_context = youtube_vectorstore.similarity_search(question)
        context = pdf_context + youtube_context  # Łączenie wyników
    
    # Łączenie tekstu z dokumentów w jeden "context"
    context_text = " ".join([doc.page_content for doc in context])
    return context_text



In [ ]:
# Placeholder dla modelu (np. OpenAI GPT)
def model(prompt):
    # Tutaj model generuje odpowiedź na podstawie prompta
    return f"Generated response based on prompt: {prompt}"

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [ ]:
# Przykład pytania użytkownika
question = "What is the mechanism of action of the drug?"
source = "both"  # Może być "pdf", "youtube" lub "both"

# Definicja chaina za pomocą itemgetter i operatorów
chain = (
    {
        # Pobieranie kontekstu z wybranego źródła na podstawie pytania
        "context": lambda data: retrieve_context(data["question"], data["source"], pdf_vectorstore, youtube_vectorstore),
        "question": itemgetter("question"),
        "source": itemgetter("source"),
    }
    | prompt  # Formatowanie prompta
    | model  # Generowanie odpowiedzi przez model
    | parser  # Parsowanie odpowiedzi
)

# Przykładowe dane wejściowe do chaina
data = {
    "question": question,
    "source": source,
}

# Wykonanie chaina
response = chain(data)
print(response)